# Dataset Analysis

Ноутбук для анализа накопленного `synthesized_pii.jsonl`: метрики, распределения, exact mismatches, похожие пары и быстрый просмотр примеров.

Запускать из папки `synthetic_data_generation/synthesizer_agent/notebooks`.

In [16]:
from pathlib import Path
import sys
import json
from pprint import pprint

import pandas as pd

SRC_DIR = Path("../src").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

from metrics_eval import load_jsonl, compute_all_metrics, _normalize_for_similarity

DATASET_PATH = Path("../outputs/synthesized_pii.jsonl")
results = load_jsonl(DATASET_PATH)
print(f"Loaded texts: {len(results)}")


## Metrics Summary

In [17]:
report = compute_all_metrics(results)

summary = {
    "dataset_stats": report["dataset_stats"],
    "tag_correctness": {
        k: v for k, v in report["tag_correctness"].items()
        if k != "exact_mismatches"
    },
    "semantic_repetition": {
        k: v for k, v in report["semantic_repetition"].items()
        if k != "nearest_similarity_values"
    },
}
pprint(summary, sort_dicts=False)


## Entity Counts

In [18]:
entity_counts_df = (
    pd.DataFrame(
        sorted(report["dataset_stats"]["entity_type_counts"].items()),
        columns=["entity", "count"]
    )
    .sort_values("count", ascending=False)
    .reset_index(drop=True)
)
entity_counts_df


## Entities Per Text

In [19]:
entities_per_text_df = pd.DataFrame(
    sorted(report["dataset_stats"]["entities_per_text"]["distribution"].items()),
    columns=["entities_per_text", "num_texts"]
)
entities_per_text_df


## Exact Mismatches

Это диагностическая проверка: `used_entities.value` должен буквально совпасть с содержимым тега. Для `NAME`, `ORGANIZATION`, `ADDRESS` расхождения часто могут быть нормальными склонениями или нормализацией адреса, поэтому это не всегда ошибка датасета.

In [ ]:
mismatches = report["tag_correctness"]["exact_mismatches"]
print(f"Exact mismatches: {len(mismatches)}")

mismatch_rows = []
for m in mismatches:
    mismatch_rows.append({
        "jsonl_line": m["jsonl_line"],
        "entity_index": m["entity_index"],
        "key": m["key"],
        "value": m["value"],
        "actual_tagged_values": " | ".join(m["actual_tagged_values"]),
    })

mismatches_df = pd.DataFrame(mismatch_rows)
mismatches_df.head(50)


In [ ]:
if not mismatches_df.empty:
    display(mismatches_df.groupby("key").size().sort_values(ascending=False).to_frame("count"))


In [ ]:
def show_mismatch(i: int):
    m = mismatches[i]
    print("=" * 100)
    print(f"Mismatch #{i}")
    print(f"JSONL line: {m['jsonl_line']}")
    print(f"Entity: {m['key']}")
    print(f"Expected value: {m['value']!r}")
    print(f"Actual tagged values: {m['actual_tagged_values']}")
    print("-" * 100)
    print(m["text"])
    print("-" * 100)
    print(json.dumps(m["used_entities"], ensure_ascii=False, indent=2))

# Example:
if mismatches:
    show_mismatch(0)


## Similar Text Pairs

Поиск ближайших похожих пар тем же способом, что и в `metrics_eval.py`: TF-IDF по символьным n-граммам после замены всех PII-тегов на `[PII]`.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

texts_norm = [_normalize_for_similarity(item["text"]) for item in results]

if len(texts_norm) > 1:
    vectorizer = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
    matrix = vectorizer.fit_transform(texts_norm)
    sim = cosine_similarity(matrix)
    np.fill_diagonal(sim, -1.0)

    pairs = []
    for i in range(sim.shape[0]):
        j = int(sim[i].argmax())
        pairs.append({
            "text_index": i,
            "jsonl_line": i + 1,
            "nearest_index": j,
            "nearest_jsonl_line": j + 1,
            "similarity": float(sim[i, j]),
        })

    pairs_df = pd.DataFrame(pairs).sort_values("similarity", ascending=False).reset_index(drop=True)
else:
    pairs_df = pd.DataFrame(columns=["text_index", "jsonl_line", "nearest_index", "nearest_jsonl_line", "similarity"])

pairs_df.head(30)


In [ ]:
def show_pair(row_idx: int):
    row = pairs_df.iloc[row_idx]
    i = int(row["text_index"])
    j = int(row["nearest_index"])
    print("=" * 100)
    print(f"Pair #{row_idx}: lines {i + 1} and {j + 1}, similarity={row['similarity']:.4f}")
    print("-" * 100)
    print(results[i]["text"])
    print("-" * 100)
    print(results[j]["text"])

# Example:
if len(pairs_df):
    show_pair(0)


## Quick Text Viewer

In [ ]:
def show_example(i: int):
    item = results[i]
    print("=" * 100)
    print(f"EXAMPLE #{i}, JSONL line {i + 1}")
    print("TEXT:")
    print(item.get("text", ""))
    print("LOGIC OF ENTRY:")
    print(item.get("logic_of_entry", ""))
    print("USED ENTITIES:")
    print(json.dumps(item.get("used_entities", []), ensure_ascii=False, indent=2))
    print("SOURCE FRAGMENT:")
    print(item.get("source_fragment", ""))

show_example(0)
